In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path('/home/swl/braincell-ion_dyn').resolve()
if str(repo_root) in sys.path:
    sys.path.remove(str(repo_root))
sys.path.insert(0, str(repo_root))

os.environ['JAX_PLATFORMS'] = 'cpu'
os.environ['CUDA_VISIBLE_DEVICES'] = ''

import brainstate
import braintools
import brainunit as u
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import braincell
from braincell import Branch, Cell, Morphology
from braincell.filter import BranchSlice, at
from braincell.mech import Ion, MechanismProbe, register_ion
from braincell.ion import Calcium
from braincell.ion._base import Conserve, Factor, KineticIon, Reaction, Species

brainstate.environ.set(precision=64)
print('braincell version:', braincell.__version__)


In [ ]:
dt_ms = 0.01
duration_ms = 20.0
temperature_celsius = 36.0
v_init_mV = -60.0
pump_tot_mM_um = 1.0
ci0_mM = 0.2
diam_um = 20.0
pumpbound0_mM_um = 0.0

print({
    'dt_ms': dt_ms,
    'duration_ms': duration_ms,
    'PumpTot': pump_tot_mM_um,
    'Ci0': ci0_mM,
    'diam_um': diam_um,
    'PumpBound0': pumpbound0_mM_um,
})


In [ ]:
mod_dir = Path(braincell.__file__).resolve().parent.parent / 'examples' / 'neuron_compare' / 'Cerebellum_mod' / 'DCN'
print((mod_dir / 'ion' / 'ToyDiamFactorKinetic_SU15_DCN.mod').exists())
from neuron import h, load_mechanisms
if not load_mechanisms(str(mod_dir.resolve())):
    raise RuntimeError('mechanisms not found')
h.load_file('stdrun.hoc')

sec = h.Section(name='soma')
sec.L = 20.0
sec.diam = diam_um
sec.nseg = 1
seg = sec(0.5)
sec.insert('ToyDiamFactorKinetic_SU15_DCN')
mech = seg.ToyDiamFactorKinetic_SU15_DCN
mech.PumpTot = pump_tot_mM_um
h.celsius = temperature_celsius
h.dt = dt_ms
h.tstop = duration_ms
h.v_init = v_init_mV
t_vec = h.Vector().record(h._ref_t)
cai_vec = h.Vector().record(seg._ref_cai)
pumpbound_vec = h.Vector().record(mech._ref_pumpbound)
pumpfree_vec = h.Vector().record(mech._ref_pumpfree)
h.finitialize(h.v_init)
seg.cai = ci0_mM
mech.pumpbound = pumpbound0_mM_um
mech.pumpfree = pump_tot_mM_um * np.pi * diam_um - pumpbound0_mM_um
h.frecord_init()
h.continuerun(h.tstop)
neuron_t_ms = np.asarray(t_vec)
neuron_cai_mM = np.asarray(cai_vec)
neuron_pumpbound = np.asarray(pumpbound_vec)
neuron_pumpfree = np.asarray(pumpfree_vec)
print('NEURON PI*diam:', np.pi * float(seg.diam))
print('NEURON end ci/pumpbound/pumpfree:', float(neuron_cai_mM[-1]), float(neuron_pumpbound[-1]), float(neuron_pumpfree[-1]))


In [ ]:
@register_ion('ToyDiamFactorKineticNotebook_SU2015_DCN')
class ToyDiamFactorKineticNotebook_SU2015_DCN(Calcium, KineticIon):
    __module__ = 'braincell.ion'

    factors = (
        Factor('pump_area', lambda self: u.math.pi * self.diam_mid),
    )
    species = (
        Species('Ci', init=0.2 * u.mM),
        Species('PumpFree', init=1.0 * u.mM * u.um, factor='pump_area'),
        Species('PumpBound', init=0.0 * u.mM * u.um, factor='pump_area'),
    )
    reactions = (
        Reaction(lhs={'Ci': 1}, rhs={'Ci': 1}, forward=lambda self, V, x: 0.0 / u.ms, backward=lambda self, V, x: 0.0 / u.ms),
    )
    sources = ()
    conserves = (
        Conserve(species=('PumpFree', 'PumpBound'), algebraic='PumpFree', total=lambda self, V, x: self.PumpTot * u.math.pi * self.diam_mid),
    )

    def __init__(self, size, *, temp=u.celsius2kelvin(36.0), PumpTot=1.0 * u.mM * u.um, Co=None, Ci_initializer=0.2 * u.mM, PumpBound_initializer=0.0 * u.mM * u.um, solver='rk4', substeps=5, name=None, **channels):
        super().__init__(size=size, name=name, **channels)
        self._init_kinetic_ion(Co=Co, temp=temp, valence=None, species_initializers={'Ci': Ci_initializer, 'PumpBound': PumpBound_initializer}, solver=solver, substeps=substeps)
        self.Ci_initializer = Ci_initializer
        self.PumpBound_initializer = PumpBound_initializer
        self.PumpTot = braintools.init.param(PumpTot, self.varshape, allow_none=False)


In [ ]:
soma = Branch.from_lengths(lengths=[20.0] * u.um, radii=[10.0, 10.0] * u.um, type='soma')
morpho = Morphology.from_root(soma, name='soma')
cell = Cell(morpho, solver='staggered')
cell.paint(
    BranchSlice(branch_index=0, prox=0.0, dist=1.0),
    Ion(
        'ToyDiamFactorKineticNotebook_SU2015_DCN',
        name='ca_diam_factor',
        temp=u.celsius2kelvin(temperature_celsius),
        PumpTot=pump_tot_mM_um * u.mM * u.um,
        Ci_initializer=ci0_mM * u.mM,
        PumpBound_initializer=pumpbound0_mM_um * u.mM * u.um,
    ),
)
cell.place(
    at('soma', 0.5),
    MechanismProbe(mechanism='ca_diam_factor', field='Ci'),
    MechanismProbe(mechanism='ca_diam_factor', field='PumpBound'),
    MechanismProbe(mechanism='ca_diam_factor', field='PumpFree'),
)
with brainstate.environ.context(precision=64):
    cell.init_state()
    runtime_ion = cell.runtime.get_ion('ca_diam_factor')
    print('BrainCell runtime diam_mid:', runtime_ion.diam_mid)
    print('BrainCell PI*diam_mid:', u.math.pi * runtime_ion.diam_mid)
    result = cell.run(dt=dt_ms * u.ms, duration=duration_ms * u.ms)

cell_cai_mM = np.asarray(result.traces['soma(0.5)_ca_diam_factor_Ci'].to_decimal(u.mM))
cell_pumpbound = np.asarray(result.traces['soma(0.5)_ca_diam_factor_PumpBound'].to_decimal(u.mM * u.um))
cell_pumpfree = np.asarray(result.traces['soma(0.5)_ca_diam_factor_PumpFree'].to_decimal(u.mM * u.um))
print('Cell end Ci/PumpBound/PumpFree:', float(cell_cai_mM[-1]), float(cell_pumpbound[-1]), float(cell_pumpfree[-1]))


In [ ]:
compare_t_ms = neuron_t_ms[1:]

print('Geometry sanity check complete.')
print('NEURON total pool should equal BrainCell total pool when factor uses PI*diam / PI*diam_mid.')
